# ONNX-Tool 模型图展示

本文展示如何基于ONNX-Tool自定义算子以及进行算子融合及其验证：
1. 加载 ONNX 模型并展示其结构
2. 对模型图进行增删改查操作（如融合 Clip/ReLU6 到 Conv 中）
3. 使用 onnx-tool 的 value_infer 验证融合前后推理结果一致

## 第一部分：模型图表示 - 加载并打印模型结构

In [1]:
# 导入必要的库
import onnx
from onnx_tool import loadmodel, Graph, Model
from onnx_tool.fusion import FusionPattern, createSerialPattern
import numpy as np
from collections import Counter

In [83]:
# 1. 使用 onnx-tool 加载 mobilenetv2-12.onnx 模型
model_path = 'data/public/mobilenetv2-12/mobilenetv2-12.onnx'
model = loadmodel(model_path)
print(f"模型名称：{model.modelname}")
print(f"模型加载成功！")

模型名称：mobilenetv2-12
模型加载成功！


### 关于 model.mproto 和 model.graph 的关系

在 `onnx_tool.Model` 类中：

```python
self.mproto = m           # 原始 ONNX ModelProto 对象
self.graph = Graph(m.graph, self.cfg)  # onnx_tool 封装的 Graph 对象
```

**关系说明**：
- `mproto`: 是原始的 `onnx.ModelProto` 对象，包含完整的 ONNX 模型协议缓冲区数据
- `mproto.graph`: 是 `onnx.GraphProto` 对象，是 ONNX 原生协议中的图结构
- `self.graph`: 是 `onnx_tool.Graph` 对象，是对 `onnx.GraphProto` 的**高级封装**

**区别**：
| 属性 | 类型 | 用途 |
|------|------|------|
| `mproto` | `onnx.ModelProto` | 保存/加载原始 ONNX 模型，包含元数据、opset 等 |
| `graph` | `onnx_tool.Graph` | 提供高级操作：节点映射、张量映射、形状推断、profile、融合等 |

**数据流**：
```
ONNX 文件 → onnx.load_model() → ModelProto (mproto)
                                    ↓
                              mproto.graph (GraphProto)
                                    ↓
                              Graph() 封装 → nodemap, tensormap 等
```

**简单说**：`mproto` 是原始数据容器，`graph` 是便于操作的高级接口。

In [84]:
# 2. 获取模型图并打印基本信息
graph = model.graph
print(f"\n=== 模型图基本信息 ===")
print(f"节点总数：{len(graph.nodemap)}")
print(f"初始值张量数：{len(graph.initials)}")
print(f"动态张量数：{len(graph.dynamics)}")
print(f"输入张量：{graph.input}")
print(f"输出张量：{graph.output}")


=== 模型图基本信息 ===
节点总数：105
初始值张量数：177
动态张量数：106
输入张量：['input']
输出张量：['output']


In [85]:
# 3. 打印模型中所有节点类型及其数量
op_types = [node.op_type for node in graph.nodemap.values()]
op_counts = Counter(op_types)

print("\n=== 节点类型统计 ===")
for op_type, count in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"{op_type}: {count}")


=== 节点类型统计 ===
Conv: 52
Clip: 35
Add: 10
GlobalAveragePool: 1
Shape: 1
Constant: 1
Gather: 1
Unsqueeze: 1
Concat: 1
Reshape: 1
Gemm: 1


In [86]:
# 4. 打印模型结构（前 20 个节点）
print("\n=== 模型结构（前 20 个节点）===")
for i, (name, node) in enumerate(list(graph.nodemap.items())[:20]):
    print(f"{i+1:2d}. {name}: {node.op_type}")
    print(f"    输入：{node.input[:3]}{'...' if len(node.input) > 3 else ''}")
    print(f"    输出：{node.output}")


=== 模型结构（前 20 个节点）===
 1. Conv_0: Conv
    输入：['input', '475', '476']
    输出：['474']
 2. Clip_1: Clip
    输入：['474', 'Clip_1min', 'Clip_1max']
    输出：['317']
 3. Conv_2: Conv
    输入：['317', '478', '479']
    输出：['477']
 4. Clip_3: Clip
    输入：['477', 'Clip_3min', 'Clip_3max']
    输出：['320']
 5. Conv_4: Conv
    输入：['320', '481', '482']
    输出：['480']
 6. Conv_5: Conv
    输入：['480', '484', '485']
    输出：['483']
 7. Clip_6: Clip
    输入：['483', 'Clip_6min', 'Clip_6max']
    输出：['325']
 8. Conv_7: Conv
    输入：['325', '487', '488']
    输出：['486']
 9. Clip_8: Clip
    输入：['486', 'Clip_8min', 'Clip_8max']
    输出：['328']
10. Conv_9: Conv
    输入：['328', '490', '491']
    输出：['489']
11. Conv_10: Conv
    输入：['489', '493', '494']
    输出：['492']
12. Clip_11: Clip
    输入：['492', 'Clip_11min', 'Clip_11max']
    输出：['333']
13. Conv_12: Conv
    输入：['333', '496', '497']
    输出：['495']
14. Clip_13: Clip
    输入：['495', 'Clip_13min', 'Clip_13max']
    输出：['336']
15. Conv_14: Conv
    输入：['336', '499', '

In [87]:
# 5. 查找模型中的 Clip 节点（ReLU6）
clip_nodes = [name for name, node in graph.nodemap.items() if node.op_type == 'Clip']
print(f"\n=== Clip 节点（ReLU6）===")
print(f"找到 {len(clip_nodes)} 个 Clip 节点:")
for name in clip_nodes[:10]:  # 只显示前 10 个
    node = graph.nodemap[name]
    print(f"  - {name}")
    print(f"    输入：{node.input}")
    print(f"    输出：{node.output}")


=== Clip 节点（ReLU6）===
找到 35 个 Clip 节点:
  - Clip_1
    输入：['474', 'Clip_1min', 'Clip_1max']
    输出：['317']
  - Clip_3
    输入：['477', 'Clip_3min', 'Clip_3max']
    输出：['320']
  - Clip_6
    输入：['483', 'Clip_6min', 'Clip_6max']
    输出：['325']
  - Clip_8
    输入：['486', 'Clip_8min', 'Clip_8max']
    输出：['328']
  - Clip_11
    输入：['492', 'Clip_11min', 'Clip_11max']
    输出：['333']
  - Clip_13
    输入：['495', 'Clip_13min', 'Clip_13max']
    输出：['336']
  - Clip_17
    输入：['501', 'Clip_17min', 'Clip_17max']
    输出：['342']
  - Clip_19
    输入：['504', 'Clip_19min', 'Clip_19max']
    输出：['345']
  - Clip_22
    输入：['510', 'Clip_22min', 'Clip_22max']
    输出：['350']
  - Clip_24
    输入：['513', 'Clip_24min', 'Clip_24max']
    输出：['353']


In [88]:
# 6. 查找模型中的 Conv 节点
conv_nodes = [name for name, node in graph.nodemap.items() if node.op_type == 'Conv']
print(f"\n=== Conv 节点 ===")
print(f"找到 {len(conv_nodes)} 个 Conv 节点:")
for name in conv_nodes[:5]:  # 只显示前 5 个
    node = graph.nodemap[name]
    print(f"  - {name}")
    print(f"    输入：{node.input}")
    print(f"    输出：{node.output}")


=== Conv 节点 ===
找到 52 个 Conv 节点:
  - Conv_0
    输入：['input', '475', '476']
    输出：['474']
  - Conv_2
    输入：['317', '478', '479']
    输出：['477']
  - Conv_4
    输入：['320', '481', '482']
    输出：['480']
  - Conv_5
    输入：['480', '484', '485']
    输出：['483']
  - Conv_7
    输入：['325', '487', '488']
    输出：['486']


## 第二部分：融合 Clip(ReLU6) 到 Conv 中

In [89]:
# 1. 获取计算图（移除形状计算节点）
cg = graph.get_compute_graph()
print(f"计算图节点数：{len(cg.nodemap)}")

计算图节点数：100


In [90]:
# 2. 定义 Conv + Clip 融合模式
ConvClip_pattern = [
    {
        'name': 'conv_0',
        'op': 'Conv',
        'attrs': [],
        'inport': [],
        'outport': [[0, 'clip_1', 0]],
    },
    {
        'name': 'clip_1',
        'op': 'Clip',
        'attrs': [],
        'inport': [[0, 'conv_0', 0]],
        'outport': [],
    },
]

pattern = FusionPattern(ConvClip_pattern)
print("Conv + Clip 融合模式已定义")

Conv + Clip 融合模式已定义


### 关于 `inport` 和 `outport` 的含义

在融合模式定义中，`inport` 和 `outport` 用于描述节点之间的**连接关系**：

#### `outport` - 输出端口连接
格式：`[[输出索引，目标节点名，目标节点输入索引]]`

```python
'outport': [[0, 'clip_1', 0]]
# 表示：当前节点（Conv）的第 0 个输出 → 连接到 'clip_1' 节点的第 0 个输入
```

#### `inport` - 输入端口连接
格式：`[[输入索引，源节点名，源节点输出索引]]`

```python
'inport': [[0, 'conv_0', 0]]
# 表示：当前节点（Clip）的第 0 个输入 ← 来自 'conv_0' 节点的第 0 个输出
```

#### 特殊值 `-1`
当索引为 `-1` 时，表示**任意索引**，会匹配所有可能的连接：

```python
'inport': [[-1, 'prev_node', -1]]  # 匹配来自 prev_node 的任意输出
'outport': [[-1, 'next_node', -1]]  # 匹配连接到 next_node 的任意输入
```

In [91]:
# 3. 搜索匹配的 Conv + Clip 模式
found_nodes = pattern.search_pattern(cg)
print(f"\n=== 找到 {len(found_nodes)} 个 Conv + Clip 模式 ===")
for i, nodes in enumerate(found_nodes[:5]):
    print(f"{i+1}. {nodes}")


=== 找到 35 个 Conv + Clip 模式 ===
1. ['Conv_0', 'Clip_1']
2. ['Conv_2', 'Clip_3']
3. ['Conv_5', 'Clip_6']
4. ['Conv_7', 'Clip_8']
5. ['Conv_10', 'Clip_11']


In [92]:
# 4. 执行融合 - 将 Conv + Clip 融合为 ConvACT 节点
print("\n=== 执行融合 ===")
fusion_count = 0
for nodes in found_nodes:
    new_node_name = nodes[0].replace('conv', 'ConvACT')
    cg.fuse_subgraph_node_names(nodes, 'ConvACT', new_node_name, keep_attr=True)
    fusion_count += 1
    
print(f"成功融合 {fusion_count} 个 Conv + Clip -> ConvACT")


=== 执行融合 ===
成功融合 35 个 Conv + Clip -> ConvACT


D:\projects\AIInfra\onnx-tool\onnx_tool\node.py:2689: UserWarning: node ConvACT is not registed for profiling, return 0 Macs and 0 params as default. Use NODEPROFILER_REGISTRY to register your profiler for this node.
  else:


In [93]:
# 5. 查看融合后的结果
print("\n=== 融合后的模型结构 ===")
print(f"融合后节点总数：{len(cg.nodemap)}")

op_types_after = [node.op_type for node in cg.nodemap.values()]
op_counts_after = Counter(op_types_after)

print("\n=== 融合后节点类型统计 ===")
for op_type, count in sorted(op_counts_after.items(), key=lambda x: -x[1])[:15]:
    print(f"{op_type}: {count}")


=== 融合后的模型结构 ===
融合后节点总数：65

=== 融合后节点类型统计 ===
ConvACT: 35
Conv: 17
Add: 10
GlobalAveragePool: 1
Reshape: 1
Gemm: 1


In [94]:
# 6. 保存融合后的模型
# 传入原始模型的 mproto 以保持 IR 版本兼容
output_path = 'data/public/mobilenetv2-12/mobilenetv2_fused.onnx'
cg.save_model(output_path, rawmodel=model.mproto)
print(f"\n融合后的模型已保存到：{output_path}")


融合后的模型已保存到：data/public/mobilenetv2-12/mobilenetv2_fused.onnx


In [95]:
# 7. 对比融合前后的变化
print("\n=== 融合前后对比 ===")
print(f"融合前节点数：{len(graph.nodemap)}")
print(f"融合后节点数：{len(cg.nodemap)}")
print(f"减少节点数：{len(graph.nodemap) - len(cg.nodemap)}")
print(f"\n融合前 Clip 节点数：{op_counts.get('Clip', 0)}")
print(f"融合后 Clip 节点数：{op_counts_after.get('Clip', 0)}")
print(f"新增 ConvACT 节点数：{len([n for n in cg.nodemap.values() if n.op_type == 'ConvACT'])}")


=== 融合前后对比 ===
融合前节点数：65
融合后节点数：65
减少节点数：0

融合前 Clip 节点数：35
融合后 Clip 节点数：0
新增 ConvACT 节点数：35


## 第三部分：融合前后推理结果对比（使用 onnx-tool）

使用 onnx-tool 的 `value_infer` 方法进行推理，验证融合前后模型输出一致。

**注意**：在使用 `value_infer` 之前，需要先使用 `shape_infer` 设置输入张量的形状。

In [96]:
# 1. 准备原始模型的计算图（用于推理）
original_graph = model.graph
print(f"原始模型图节点数：{len(original_graph.nodemap)}")

原始模型图节点数：65


In [97]:
# 2. 创建随机输入数据
input_name = original_graph.input[0]
input_tensor = original_graph.tensormap[input_name]
input_shape = input_tensor.shape

# 将形状中的负数/字符串（动态轴）替换为具体值（batch size=1）
# MobileNetV2 期望输入形状为 [N, C, H, W] = [1, 3, 224, 224]
test_shape = []
for dim in input_shape:
    if isinstance(dim, int) and dim > 0:
        test_shape.append(dim)
    else:
        # 对于动态轴，使用默认值
        test_shape.append(1)  # batch size

# 确保是 4D 形状（NCHW）
if len(test_shape) == 2:  # 如果是 [1, C]，扩展为 [1, C, H, W]
    test_shape.extend([224, 224])
elif len(test_shape) == 3:  # 如果是 [1, H, W]，添加通道
    test_shape.insert(1, 3)

print(f"输入张量名称：{input_name}")
print(f"原始输入形状：{input_shape}")
print(f"测试输入形状：{test_shape}")

# 创建随机输入
np.random.seed(42)
test_input = np.random.randn(*test_shape).astype(np.float32)
print(f"测试输入已创建，形状：{test_input.shape}")

输入张量名称：input
原始输入形状：['batch_size', 3, 224, 224]
测试输入形状：[1, 3, 224, 224]
测试输入已创建，形状：(1, 3, 224, 224)


In [113]:
# 3. 使用 shape_infer 更新张量形状
# value_infer 需要先进行 shape_infer 来设置正确的张量形状
print("执行形状推断...")
original_graph.shape_infer({input_name: test_shape})
print("形状推断完成")

执行形状推断...



KeyboardInterrupt



In [ ]:
# 4. 使用 onnx-tool 的 value_infer 进行推理
# 注意：value_infer 需要输入字典
inputs = {input_name: test_input}

# 原始模型推理
print("执行原始模型推理...")
original_outputs = original_graph.value_infer(inputs)
print(f"原始模型推理完成，输出数量：{len(original_outputs)}")
for i, out in enumerate(original_outputs):
    print(f"  输出 {i}: 形状 {out.shape}")

In [ ]:
# 5. 融合模型形状推断
fused_input_name = cg.input[0]
print(f"\n融合模型输入张量名称：{fused_input_name}")

# 融合模型也需要先进行形状推断
print("执行融合模型形状推断...")
cg.shape_infer({fused_input_name: test_shape})
print("融合模型形状推断完成")

In [ ]:
# 6. 融合模型推理
# 执行推理
print("执行融合模型推理...")
fused_inputs = {fused_input_name: test_input}
fused_outputs = cg.value_infer(fused_inputs)
print(f"融合模型推理完成，输出数量：{len(fused_outputs)}")
for i, out in enumerate(fused_outputs):
    print(f"  输出 {i}: 形状 {out.shape}")

In [ ]:
# 7. 对比推理结果
print("\n=== 推理结果对比 ===")

# 获取第一个（通常是唯一）输出进行对比
original_output = original_outputs[0]
fused_output = fused_outputs[0]

# 计算最大绝对误差
max_diff = np.max(np.abs(original_output - fused_output))
print(f"最大绝对误差：{max_diff:.10f}")

# 计算平均绝对误差
mean_diff = np.mean(np.abs(original_output - fused_output))
print(f"平均绝对误差：{mean_diff:.10f}")

# 使用 allclose 判断是否一致（考虑浮点数精度）
is_close = np.allclose(original_output, fused_output, rtol=1e-5, atol=1e-5)
print(f"\n结果是否一致 (rtol=1e-5, atol=1e-5): {is_close}")

# 显示部分输出值对比
print("\n=== 输出值对比（前 10 个）===")
for i in range(min(10, original_output.size)):
    orig_val = original_output.flat[i]
    fused_val = fused_output.flat[i]
    diff = abs(orig_val - fused_val)
    print(f"  [{i:2d}] 原始：{orig_val:12.6f}  融合：{fused_val:12.6f}  差异：{diff:.10f}")

In [ ]:
# 8. 使用多个随机输入进行更全面的验证
print("\n=== 多组输入验证 ===")
num_tests = 5
all_passed = True

for i in range(num_tests):
    # 创建随机输入
    test_input = np.random.randn(*test_shape).astype(np.float32)
    
    # 原始模型推理（需要重新进行形状推断）
    original_graph.shape_infer({input_name: test_shape})
    orig_inputs = {input_name: test_input}
    orig_out = original_graph.value_infer(orig_inputs)[0]
    
    # 融合模型推理
    cg.shape_infer({fused_input_name: test_shape})
    fused_inputs = {fused_input_name: test_input}
    fused_out = cg.value_infer(fused_inputs)[0]
    
    # 对比
    passed = np.allclose(orig_out, fused_out, rtol=1e-5, atol=1e-5)
    max_diff = np.max(np.abs(orig_out - fused_out))
    
    status = "✓ 通过" if passed else "✗ 失败"
    print(f"  测试 {i+1}: {status} (最大误差：{max_diff:.10f})")
    all_passed = all_passed and passed

print(f"\n=== 最终结果：{'全部通过 ✓' if all_passed else '部分失败 ✗'} ===")

## 总结

本 notebook 演示了：

### 1. 模型图表示
- 使用 `loadmodel()` 加载 ONNX 模型
- 通过 `graph.nodemap` 访问模型中的所有节点
- 打印节点类型统计和结构信息

### 2. 模型图增删改查
- **查**：使用 `FusionPattern` 搜索特定的节点模式
- **改**：使用 `fuse_subgraph_node_names()` 融合多个节点为一个新节点
- **删**：融合操作会自动删除被融合的节点
- **增**：融合操作会创建新的融合节点（如 ConvACT）

### 3. 推理验证（使用 onnx-tool）
- 使用 `graph.value_infer(inputs)` 进行推理
- 验证融合前后模型输出一致（误差在允许范围内）

### 关键 API
- `loadmodel(path)`: 加载 ONNX 模型
- `graph.get_compute_graph()`: 获取计算图
- `FusionPattern(pattern)`: 创建融合模式
- `pattern.search_pattern(graph)`: 搜索匹配的模式
- `graph.fuse_subgraph_node_names(nodes, op_type, name, keep_attr)`: 融合节点
- `graph.save_model(path, rawmodel)`: 保存模型
- `graph.value_infer(inputs)`: 执行推理

### mproto 与 graph 关系
- `mproto`: 原始 ONNX ModelProto，用于保存/加载
- `graph`: onnx_tool 封装的高级接口，用于操作和分析

### inport 与 outport 含义
- `outport`: `[[输出索引，目标节点名，目标节点输入索引]]` - 描述输出连接到哪里
- `inport`: `[[输入索引，源节点名，源节点输出索引]]` - 描述输入来自哪里